# Xovato Model Test

Downloads each CPU model, runs a **real** inference sanity check, and prints size,
latency and an accuracy signal. Runs in Colab (GPU) or locally (CPU) - the same
`app/` code either way.

| Model | Feature | What this notebook checks |
| --- | --- | --- |
| U²-Net-p | Background removal | mask coverage + latency on a test subject |
| YuNet | Face detection | face count + landmarks on your image |
| GPEN-BFR-256 | Face restoration | restored 256² face + latency |
| Audio DSP | Audio enhancement | LUFS/peak before→after + realtime factor |
| Image | Image enhancement | measured corrections + latency |

See [../models/README.md](README.md) for the reported accuracy of each model.


## 1. Get the code and dependencies


In [ ]:
import os, sys, subprocess, platform, time, io, urllib.request
from pathlib import Path

# In Colab, clone the repo; locally, assume we are inside it.
REPO = 'https://github.com/Zubair-hussain/Xovato-cpu-ai-studio.git'
if Path('app').exists():
    ROOT = Path('.').resolve()
elif Path('../app').exists():
    ROOT = Path('..').resolve()
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=True)
    ROOT = Path('Xovato-cpu-ai-studio').resolve()
sys.path.insert(0, str(ROOT))
print('repo root:', ROOT)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pillow', 'numpy', 'onnxruntime', 'soundfile'], check=False)
print('python', platform.python_version())

## 2. Download the small ONNX models (size-checked)


In [ ]:
MODELS = ROOT / 'models'
DOWNLOADS = [
    ('segmentation/u2netp.onnx', 3_000_000,
     'https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2netp.onnx'),
    ('face/yunet.onnx', 100_000,
     'https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'),
    ('face/gpen_bfr_256.onnx', 20_000_000,
     'https://github.com/facefusion/facefusion-assets/releases/download/models-3.0.0/gpen_bfr_256.onnx'),
]
for rel, min_bytes, url in DOWNLOADS:
    dest = MODELS / rel
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print(f'{rel}: present ({dest.stat().st_size/1e6:.1f} MB)'); continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        print(f'{rel}: downloading ...', end=' ')
        urllib.request.urlretrieve(url, dest)
        ok = dest.stat().st_size >= min_bytes
        print('ok' if ok else 'TOO SMALL (LFS pointer?)', f'{dest.stat().st_size/1e6:.1f} MB')
    except Exception as e:
        print('failed:', e)

## 3. Background removal (U²-Net-p)


In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from app.config import get_settings
from app import image_processing as ip
settings = get_settings()

# A salient subject on a cluttered background.
rng = np.random.default_rng(0)
img = Image.fromarray((rng.integers(80, 200, (600, 600, 3))).astype('uint8'))
d = ImageDraw.Draw(img)
d.ellipse([250, 120, 350, 220], fill=(235, 205, 180))
d.rectangle([230, 220, 370, 500], fill=(200, 70, 70))
buf = io.BytesIO(); img.save(buf, format='JPEG')

model = settings.segmentation_model_path
if ip.segmentation_model_available(model):
    t = time.perf_counter()
    payload, analysis = ip.remove_background(buf.getvalue(), model_path=model, output_format='png')
    dt = time.perf_counter() - t
    print(f'latency {dt*1000:.0f} ms  |  {analysis.summary}')
    display(Image.open(io.BytesIO(payload)))
else:
    print('u2netp.onnx missing - re-run cell 2')

## 4. Face detection + restoration (YuNet + GPEN)

Upload a **real face photo** (Colab: Files panel) and set its path - the models
correctly find no face in synthetic shapes.


In [ ]:
FACE_IMAGE = ''  # e.g. '/content/portrait.jpg'
from app.face_restoration import detect_faces, face_models_available
from app import image_processing as ip
if FACE_IMAGE and Path(FACE_IMAGE).exists() and face_models_available(settings.face_detector_path, settings.face_restorer_path):
    data = Path(FACE_IMAGE).read_bytes()
    image = ip.load_image(data)
    faces = detect_faces(image, settings.face_detector_path)
    print(f'YuNet found {len(faces)} face(s); top score '
          f'{max((f.score for f in faces), default=0):.2f}')
    t = time.perf_counter()
    payload, analysis = ip.restore_faces_image(data, detector_path=settings.face_detector_path,
                                               restorer_path=settings.face_restorer_path)
    print(f'restored {analysis.faces_detected} face(s) in {time.perf_counter()-t:.2f}s')
    display(Image.open(io.BytesIO(payload)))
else:
    print('Set FACE_IMAGE to a real portrait and ensure yunet.onnx + gpen_bfr_256.onnx are present.')

## 5. Audio enhancement (CPU DSP)


In [ ]:
import soundfile as sf
from app import audio_processing as ap
sr = 44100; t = np.arange(sr*10)/sr
voice = (0.5*np.sin(2*np.pi*140*t)+0.25*np.sin(2*np.pi*280*t)) * (np.sin(2*np.pi*3*t)**8)
noisy = np.stack([0.35*voice + 0.08*rng.standard_normal(t.size) + 0.05*np.sin(2*np.pi*50*t)]*2, axis=1).astype('float32')
b = io.BytesIO(); sf.write(b, noisy, sr, format='WAV', subtype='FLOAT')
start = time.perf_counter()
payload, fmt, analysis = ap.enhance_audio(b.getvalue(), target_lufs=-16.0)
dt = time.perf_counter() - start
print(analysis.summary)
print(f'{10/dt:.1f}x realtime on this machine')
try:
    from IPython.display import Audio, display
    display(Audio(b.getvalue()), Audio(payload))
except Exception: pass

## 6. Image enhancement (CPU)


In [ ]:
from app import image_processing as ip
flat = Image.fromarray((np.asarray(img, dtype='float32')*0.5 + 90).clip(0,255).astype('uint8'))
fb = io.BytesIO(); flat.save(fb, format='JPEG', quality=90)
t = time.perf_counter()
payload, analysis = ip.enhance_image(fb.getvalue())
print(f'{(time.perf_counter()-t)*1000:.0f} ms  |  {analysis.summary}')
display(flat, Image.open(io.BytesIO(payload)))

## 7. Text to speech (optional, heavy)

Needs the ~2.5 GB OmniVoice checkpoint and the `omnivoice` package. Fast on GPU,
slow on CPU. Left as a manual step:


In [ ]:
# !pip install -q omnivoice huggingface_hub
# from huggingface_hub import snapshot_download
# snapshot_download('k2-fsa/OmniVoice', local_dir=str(MODELS/'omnivoice'/'k2-fsa_OmniVoice'))
# os.environ['ENHANCEAI_USE_REAL_OMNIVOICE'] = 'true'
# from app.voice_engine import voice_engine
# from app.voice_styles import resolve_style
# r = voice_engine.synthesize(get_settings(), text='Hello from a local model.', style=resolve_style('calm'))
# import soundfile as sf; sf.write('tts.wav', r.audio, r.sample_rate)
print('TTS test is manual - uncomment above once the checkpoint is downloaded.')